In [54]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, mean_squared_error, r2_score, mean_absolute_error, confusion_matrix

In [55]:
#Load Data
base_path = os.path.dirname(os.getcwd()) 
data_path = os.path.join(base_path, 'data', 'pipe_condition_class_synthetic.csv')

df = pd.read_csv(data_path)
print(df.head())
df.info()

#Create Features and Target Variable
x = df.drop('Condition Rating', axis = 1)
y = df['Condition Rating']


   Age Material  Diameter  Slope  Depth      Length  Soil PH Soil Type  \
0   57      VCP         8   0.34  11.32  221.045734      5.7      Rock   
1   35      VCP        12   0.40   7.00  342.975986      6.8      Clay   
2   28      PVC         8   0.50   8.58  295.072420      5.4      Sand   
3   31      PVC        15   0.17  10.50  492.085601      8.2      Clay   
4   58      PVC         8   0.36   8.58  334.559947      8.2      Clay   

  Road Type  Condition Rating  
0    Street                 3  
1    Street                 5  
2    Street                 1  
3    Street                 1  
4     Alley                 3  
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               30000 non-null  int64  
 1   Material          30000 non-null  str    
 2   Diameter          30000 non-null  int64  
 3   Slope             30000 n

In [56]:
#Split Categorical and Numerical Features
numerical_cols = ['Age','Diameter','Slope','Depth','Length','Soil PH']
categorical_cols = ['Material','Road Type','Soil Type']

#Training and Testing Datasets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (24000, 9)
Testing features shape: (6000, 9)


In [57]:
#Imputation & Encoding, Scaling, and Model Pipeline
numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy= 'mean')),
    ('scaler',StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

#Join Pipeline
preprocessor = ColumnTransformer([
    ('num',numerical_pipeline,numerical_cols),
    ('cat',categorical_pipeline,categorical_cols)
])

#Model Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier())
])

param_grid = [
    {
        'model': [RandomForestClassifier(class_weight='balanced', random_state=42)],
        'model__max_depth': [5, 10, 20, None],
        'model__n_estimators': [100]
    },
    {
        'model': [HistGradientBoostingClassifier(class_weight='balanced', random_state=42)],
        'model__max_depth': [5, 10, 20, None]
    }
]

grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=2  
)

In [58]:
#Train and Model
grid_search.fit(X_train, y_train)
prediction = grid_search.predict(X_test)

        
print("Best Model and Parameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

#View Encoding
encoded_cols = grid_search.best_estimator_.named_steps['preprocessor'].transformers_[1][1].named_steps['encoder'].get_feature_names_out(categorical_cols)
print("Encoded Categorical Columns:", encoded_cols)

#Evaluate Model
winning_model_name = grid_search.best_estimator_.named_steps['model'].__class__.__name__

print(f"--- Metrics for Best Model: {winning_model_name} ---")
print("Best Hyperparameters:", grid_search.best_params_)

print("\nClassification Report:\n", classification_report(y_test, prediction))
print("Confusion Matrix:\n", confusion_matrix(y_test, prediction))
print("Accuracy Score:", accuracy_score(y_test, prediction))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Model and Parameters: {'model': RandomForestClassifier(class_weight='balanced', random_state=42), 'model__max_depth': 10, 'model__n_estimators': 100}
Best Cross-Validation Score: 0.2784481093378998
Encoded Categorical Columns: ['Material_PVC' 'Material_RC' 'Material_VCP' 'Road Type_Alley'
 'Road Type_Easement' 'Road Type_Highway' 'Road Type_Street'
 'Soil Type_Clay' 'Soil Type_Gravel' 'Soil Type_Loam' 'Soil Type_Rock'
 'Soil Type_Sand']
--- Metrics for Best Model: RandomForestClassifier ---
Best Hyperparameters: {'model': RandomForestClassifier(class_weight='balanced', random_state=42), 'model__max_depth': 10, 'model__n_estimators': 100}

Classification Report:
               precision    recall  f1-score   support

           1       0.67      0.66      0.67      2958
           2       0.04      0.04      0.04       265
           3       0.47      0.32      0.38      2289
           4       0.07      0.15      0.10    